# Prepare Dashboard Data

Runs all three policies (RL, rule-based, random) through the full 24-hour simulation, capturing every frame, and exports everything into ONE lightweight JSON file. This keeps the actual dashboard (Streamlit app) fast and simple -- it just loads this JSON and renders it, without needing to reload the road network, rasters, or trained model at dashboard runtime.

**Run this in Colab. Then DOWNLOAD the resulting `dashboard_data.json` file to your own laptop** (from Drive: `DisasterTwin_Project/results/dashboard_data.json`) -- you'll need it sitting next to the Streamlit app file to run the dashboard locally.

---
## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/DisasterTwin_Project'

!pip install osmnx networkx rasterio geopandas pandas numpy gymnasium stable-baselines3 -q
print('Setup ready.')

In [ ]:
import sys, os
code_dir = f'{PROJECT_DIR}/code'
sys.path.append(code_dir)
from digital_twin import DigitalTwin
from disaster_env_v3 import DisasterAllocationEnvV3, RESOURCE_TYPES, TOP_K_REQUESTS
from rule_based_allocator import RuleBasedAllocator

import osmnx as ox
import pandas as pd
import rasterio
import numpy as np
from stable_baselines3 import PPO

G = ox.load_graphml(f'{PROJECT_DIR}/data/raw/chennai_road_network.graphml')
resources_df = pd.read_csv(f'{PROJECT_DIR}/data/raw/chennai_resources.csv')

with rasterio.open(f'{PROJECT_DIR}/data/processed/chennai_flood_severity_combined.tif') as src:
    flood_severity_raster = src.read(1)
    flood_transform = src.transform
    flood_bounds = src.bounds

with rasterio.open(f'{PROJECT_DIR}/data/processed/chennai_population.tif') as src:
    population_raster = src.read(1)

demand_df = pd.read_csv(f'{PROJECT_DIR}/data/processed/chennai_demand_timeseries.csv')
demand_df['nearest_node'] = ox.distance.nearest_nodes(G, demand_df['lon'].values, demand_df['lat'].values)

def fast_step(self):
    if self.current_timestep > self.max_timestep:
        return False
    step_demand = self.demand_df[self.demand_df['timestep'] == self.current_timestep]
    for _, row in step_demand.iterrows():
        new_request = {
            'id': self._request_id_counter, 'lat': row['lat'], 'lon': row['lon'],
            'severity': row['severity_factor'], 'requests_count': row['estimated_requests'],
            'created_at': self.current_timestep, 'status': 'pending',
            'nearest_node': row['nearest_node'], 'assigned_resource': None
        }
        self.active_requests.append(new_request)
        self.request_log.append(new_request.copy())
        self._request_id_counter += 1
    self.current_timestep += 1
    return True

DigitalTwin.step = fast_step

rl_model = PPO.load(f'{PROJECT_DIR}/models/ppo_disaster_allocation_v3b.zip', device='cpu')
print('Everything loaded.')

---
## Section 1 — Downsample Static Layers (flood raster, road network) for a Lightweight Dashboard

In [ ]:
# Downsample the flood raster to a small grid (e.g. 60x60) -- plenty for a background heatmap,
# and keeps the JSON file small and the dashboard fast to render.
from scipy.ndimage import zoom

DOWNSAMPLE_SIZE = 60
zoom_factor = DOWNSAMPLE_SIZE / max(flood_severity_raster.shape)
flood_small = zoom(flood_severity_raster, zoom_factor, order=1)

print('Downsampled flood raster shape:', flood_small.shape)

# Sample a subset of road edges for a lightweight background road-network sketch
nodes, edges = ox.graph_to_gdfs(G)
MAX_EDGES = 3000
edges_sample = edges.sample(min(MAX_EDGES, len(edges)), random_state=42)

edge_lines = []
for geom in edges_sample.geometry:
    if geom is None:
        continue
    coords = list(geom.coords)
    edge_lines.append({'lons': [c[0] for c in coords], 'lats': [c[1] for c in coords]})

print(f'Sampled {len(edge_lines)} road edges for the dashboard background.')

---
## Section 2 — Run All Three Policies, Capturing Frames

In [ ]:
def make_env():
    return DisasterAllocationEnvV3(G, resources_df, demand_df, flood_severity_raster, flood_transform, population_raster)

def capture_rl_frames():
    env = make_env()
    obs, info = env.reset()
    frames = []
    resolved_lats, resolved_lons = [], []
    terminated = truncated = False
    while not (terminated or truncated):
        pending_before = {r['id']: r for r in env.twin.get_pending_requests()}
        action, _ = rl_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        pending_after_ids = {r['id'] for r in env.twin.get_pending_requests()}
        newly_resolved = set(pending_before.keys()) - pending_after_ids
        for rid in newly_resolved:
            resolved_lats.append(pending_before[rid]['lat'])
            resolved_lons.append(pending_before[rid]['lon'])
        current_pending = env.twin.get_pending_requests()
        summary = env.twin.get_state_summary()
        frames.append({
            'timestep': info['timestep'],
            'pending_lats': [r['lat'] for r in current_pending],
            'pending_lons': [r['lon'] for r in current_pending],
            'resolved_lats': list(resolved_lats),
            'resolved_lons': list(resolved_lons),
            'n_pending': len(current_pending),
            'n_resolved_total': len(resolved_lats),
            'utilization': summary['resource_utilization'],
        })
    return frames

def capture_rule_frames():
    twin = DigitalTwin(G, resources_df, demand_df, flood_severity_raster, flood_transform, population_raster)
    twin.step()
    allocator = RuleBasedAllocator(twin)
    frames = []
    resolved_lats, resolved_lons = [], []
    still_running = True
    while still_running:
        pending_before = {r['id']: r for r in twin.get_pending_requests()}
        allocator.act()
        pending_after_ids = {r['id'] for r in twin.get_pending_requests()}
        newly_resolved = set(pending_before.keys()) - pending_after_ids
        for rid in newly_resolved:
            resolved_lats.append(pending_before[rid]['lat'])
            resolved_lons.append(pending_before[rid]['lon'])
        current_pending = twin.get_pending_requests()
        summary = twin.get_state_summary()
        frames.append({
            'timestep': twin.current_timestep,
            'pending_lats': [r['lat'] for r in current_pending],
            'pending_lons': [r['lon'] for r in current_pending],
            'resolved_lats': list(resolved_lats),
            'resolved_lons': list(resolved_lons),
            'n_pending': len(current_pending),
            'n_resolved_total': len(resolved_lats),
            'utilization': summary['resource_utilization'],
        })
        still_running = twin.step()
    return frames

def capture_random_frames():
    env = make_env()
    obs, info = env.reset()
    frames = []
    resolved_lats, resolved_lons = [], []
    terminated = truncated = False
    while not (terminated or truncated):
        pending_before = {r['id']: r for r in env.twin.get_pending_requests()}
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        pending_after_ids = {r['id'] for r in env.twin.get_pending_requests()}
        newly_resolved = set(pending_before.keys()) - pending_after_ids
        for rid in newly_resolved:
            resolved_lats.append(pending_before[rid]['lat'])
            resolved_lons.append(pending_before[rid]['lon'])
        current_pending = env.twin.get_pending_requests()
        summary = env.twin.get_state_summary()
        frames.append({
            'timestep': info['timestep'],
            'pending_lats': [r['lat'] for r in current_pending],
            'pending_lons': [r['lon'] for r in current_pending],
            'resolved_lats': list(resolved_lats),
            'resolved_lons': list(resolved_lons),
            'n_pending': len(current_pending),
            'n_resolved_total': len(resolved_lats),
            'utilization': summary['resource_utilization'],
        })
    return frames

print('Running RL policy...')
rl_frames = capture_rl_frames()
print('Running rule-based policy...')
rule_frames = capture_rule_frames()
print('Running random policy...')
random_frames = capture_random_frames()
print('All three policies captured:', len(rl_frames), len(rule_frames), len(random_frames), 'frames respectively.')

---
## Section 3 — Export Everything to One JSON File

In [ ]:
import json

dashboard_data = {
    'meta': {
        'city': 'Chennai',
        'n_timesteps': len(rl_frames),
        'extent': [flood_bounds.left, flood_bounds.right, flood_bounds.bottom, flood_bounds.top],
    },
    'flood_raster_small': flood_small.tolist(),
    'road_edges': edge_lines,
    'resources': resources_df[['lat', 'lon', 'type']].to_dict('records'),
    'policies': {
        'RL Agent (PPO)': rl_frames,
        'Rule-Based Baseline': rule_frames,
        'Random': random_frames,
    }
}

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
output_path = f'{PROJECT_DIR}/results/dashboard_data.json'
with open(output_path, 'w') as f:
    json.dump(dashboard_data, f)

size_mb = os.path.getsize(output_path) / (1024*1024)
print(f'Saved dashboard data to: {output_path}')
print(f'File size: {size_mb:.2f} MB')
print('\\nNEXT STEP: download this file from Google Drive to your laptop,')
print('place it in the SAME folder as app.py, then run the Streamlit dashboard.')